# 03 — Modelling: Nightly Listing-Price Prediction

**Project:** Property Investment Intelligence System (BIU DS23)
**Owner:** Idan Gilad — models and quantitative evaluation
**Status:** Template. Runs end to end on synthetic data generated in memory (`USE_SYNTHETIC = True`) until the first real snapshot is processed. No real results.

## Purpose

Train and compare models that predict a listing's **nightly listing price** (the Inside Airbnb `price` field, local currency per night) against the charter baseline, on one shared held-out split.

## Inputs and outputs

| | Location |
| --- | --- |
| Input | Processed **Listings** table from `02_preprocessing` in `data/processed/` (schema: `data/data_dictionary.md`) |
| Output | Model artifacts and metadata in `models/` (excluded from Git) |
| Output | Split manifest in `evaluation/splits/` for `04_model_evaluation` |
| Output | Held-out **Predictions** table (dictionary schema) in `models/` |

## Guardrails

- The target is the nightly **listing price**, not acquisition value, realized revenue or occupancy.
- Grain is city + snapshot date + listing id. The same listing id must not appear on both sides of the split.
- All preprocessing and aggregations are fitted on training data only.
- Every approach is evaluated on the **same** held-out cases.

## 1. Setup

In [9]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TODO: pin versions in requirements.txt once the library set is agreed (team baseline: Python 3.11)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [11]:
# Paths (notebook lives in notebooks/, so the repo root is one level up)
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
SPLITS_DIR = REPO_ROOT / "evaluation" / "splits"

# ---- Data source switch --------------------------------------------------
# True  -> synthetic Listings data generated in memory below (no file is read or written)
# False -> the real processed Listings table produced by 02_preprocessing
USE_SYNTHETIC = True
REAL_DATA_FILE = DATA_DIR / "listings_processed.csv"

# TODO: fill from data/sources.md once a real snapshot is chosen
CITY = "dummy_city" if USE_SYNTHETIC else "TODO"
CURRENCY = "USD" if USE_SYNTHETIC else "TODO"


def make_synthetic_listings(n_listings=400, seed=RANDOM_SEED):
    """Synthetic Listings table using the field names in data/data_dictionary.md.

    Grain: snapshot_date + id (single city). Two snapshots, ~75% of listings repeated,
    raw-style price strings and bathrooms_text, so parsing and the grouped split are exercised.
    """
    rng = np.random.default_rng(seed)
    neighbourhoods = ["Downtown", "Old Town", "Waterfront", "Riverside",
                      "North End", "Hill District", "Market Street"]
    hood_premium = dict(zip(neighbourhoods, [1.35, 1.2, 1.4, 1.0, 0.9, 0.85, 1.1]))
    room_types = ["Entire home/apt", "Private room", "Shared room", "Hotel room"]
    room_premium = {"Entire home/apt": 1.0, "Private room": 0.55,
                    "Shared room": 0.35, "Hotel room": 0.8}
    bath_choices = ["1 bath", "1.5 baths", "2 baths", "1 shared bath",
                    "1 private bath", "Half-bath", "2.5 baths"]

    base = pd.DataFrame({
        "id": np.arange(1, n_listings + 1),
        "latitude": rng.uniform(40.55, 40.85, n_listings),
        "longitude": rng.uniform(-74.10, -73.85, n_listings),
        "neighbourhood_cleansed": rng.choice(neighbourhoods, n_listings),
        "room_type": rng.choice(room_types, n_listings, p=[0.55, 0.35, 0.05, 0.05]),
        "accommodates": rng.integers(1, 7, n_listings),
        "bedrooms": rng.integers(0, 4, n_listings).astype(float),
        "bathrooms_text": rng.choice(bath_choices, n_listings),
    })
    base.loc[rng.random(n_listings) < 0.05, "bedrooms"] = np.nan  # realistic gaps

    # Price with real structure so the models have something to learn
    signal = (60 + 25 * base["accommodates"]
              * base["neighbourhood_cleansed"].map(hood_premium)
              * base["room_type"].map(room_premium))
    base_price = signal * rng.lognormal(0, 0.25, n_listings)

    snapshots = []
    for snap, frac in [("2026-06-15", 1.0), ("2026-09-15", 0.75)]:
        idx = base.sample(frac=frac, random_state=seed).index
        s = base.loc[idx].copy()
        s["snapshot_date"] = snap
        price = base_price.loc[idx] * rng.lognormal(0, 0.05, len(s))
        s["price"] = ["${:,.2f}".format(p) for p in price]  # raw Inside Airbnb style
        snapshots.append(s)
    return pd.concat(snapshots, ignore_index=True)


print(f"Data source: {'SYNTHETIC (in memory)' if USE_SYNTHETIC else REAL_DATA_FILE}")

Data source: SYNTHETIC (in memory)


## 2. Load the Listings table and check it against the data dictionary

The dictionary defines the Listings grain as **city + snapshot date + listing ID** and requires currency parsing and a positive price.
With synthetic data, the table is generated in memory using the dictionary's field names; no CSV is needed.
`price` may still be a raw string such as `"$1,234.00"`, and bathrooms arrive as text (`bathrooms_text`, e.g. `"1.5 shared baths"`), so both are parsed here if `02_preprocessing` has not already done it.

In [12]:
if USE_SYNTHETIC:
    df = make_synthetic_listings()
    DATA_SOURCE = "synthetic_in_memory"
else:
    if not REAL_DATA_FILE.exists():
        raise FileNotFoundError(
            f"{REAL_DATA_FILE} not found. Run 02_preprocessing first, "
            "or set USE_SYNTHETIC = True to work with synthetic data."
        )
    df = pd.read_csv(REAL_DATA_FILE)
    DATA_SOURCE = REAL_DATA_FILE.name

print(df.shape)
df.head()

(700, 10)


,id,latitude,longitude,neighbourhood_cleansed,room_type,accommodates,bedrooms,bathrooms_text,snapshot_date,price
0,210,40.701899,-73.989978,Market Street,Entire home/apt,5,1.0,1 private bath,2026-06-15,$292.32
1,281,40.563043,-74.083208,Waterfront,Entire home/apt,3,3.0,2.5 baths,2026-06-15,$165.47
2,34,40.661138,-73.890505,Market Street,Shared room,4,0.0,Half-bath,2026-06-15,$79.83
3,211,40.758279,-73.961724,Old Town,Private room,3,0.0,1.5 baths,2026-06-15,$120.46
4,94,40.664306,-74.020946,Waterfront,Entire home/apt,6,1.0,2 baths,2026-06-15,$342.85


In [14]:
# Fields from data/data_dictionary.md -> Listings
ID_COL = "id"
SNAPSHOT_COL = "snapshot_date"
TARGET = "price"
DICTIONARY_LISTING_FIELDS = [
    ID_COL, "latitude", "longitude", "neighbourhood_cleansed", "room_type",
    "accommodates", "bedrooms", "bathrooms_text", TARGET,
]

missing = [c for c in DICTIONARY_LISTING_FIELDS + [SNAPSHOT_COL] if c not in df.columns]
assert not missing, (
    f"Columns missing from the Listings table: {missing}. "
    "Check the output of 02_preprocessing against data/data_dictionary.md."
)


def parse_price(s):
    """'$1,234.00' -> 1234.0; numeric values pass through."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    return pd.to_numeric(s.astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
                         errors="coerce")


def parse_bathrooms(text):
    """'1.5 shared baths' -> 1.5; 'Half-bath' -> 0.5; missing -> NaN."""
    if pd.isna(text):
        return np.nan
    t = str(text).lower()
    if "half" in t and not re.search(r"\d", t):
        return 0.5
    m = re.search(r"\d+(\.\d+)?", t)
    return float(m.group()) if m else np.nan


df[TARGET] = parse_price(df[TARGET])
df[SNAPSHOT_COL] = pd.to_datetime(df[SNAPSHOT_COL])
df["bathrooms"] = df["bathrooms_text"].map(parse_bathrooms)
df["bathrooms_shared"] = df["bathrooms_text"].astype(str).str.lower().str.contains("shared").astype(int)

# Dictionary validation: unique composite key and positive price
key = [c for c in ["city", SNAPSHOT_COL, ID_COL] if c in df.columns]
dupes = df.duplicated(key)
assert not dupes.any(), f"{dupes.sum()} duplicate rows on key {key}"

bad_price = df[TARGET].isna() | (df[TARGET] <= 0)
print(f"Rows removed for missing/non-positive price: {bad_price.sum()}")
df = df.loc[~bad_price].reset_index(drop=True)

print(f"rows: {len(df):,}  listings: {df[ID_COL].nunique():,}  snapshots: {df[SNAPSHOT_COL].nunique()}")

Rows removed for missing/non-positive price: 0
rows: 700  listings: 400  snapshots: 2


## 3. Target and features

**Core features** come from the Listings fields in `data/data_dictionary.md`; `bathrooms` and `bathrooms_shared` are derived from `bathrooms_text`.

**Optional features** are common Inside Airbnb columns that are **not yet in the dictionary**. They are used only if present in the processed table, and must be added to the dictionary (formula, availability at inference, units, missing-value policy) before any result relies on them.

Excluded by design: `price` itself, calendar prices, availability, any occupancy, revenue, acquisition-value or ROI column (none is established in the dictionary).

In [ ]:
CORE_NUMERIC = ["accommodates", "bedrooms", "bathrooms", "bathrooms_shared", "latitude", "longitude"]
CORE_CATEGORICAL = ["neighbourhood_cleansed", "room_type"]

# TODO: add to data_dictionary.md (with the data owner) before relying on these
OPTIONAL_NUMERIC = ["beds", "minimum_nights", "number_of_reviews", "review_scores_rating"]
OPTIONAL_CATEGORICAL = ["property_type"]

NUMERIC_FEATURES = CORE_NUMERIC + [c for c in OPTIONAL_NUMERIC if c in df.columns]
CATEGORICAL_FEATURES = CORE_CATEGORICAL + [c for c in OPTIONAL_CATEGORICAL if c in df.columns]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

FORBIDDEN = {TARGET, "estimated_revenue_l365d", "estimated_occupancy_l365d",
             "availability_30", "availability_60", "availability_90", "availability_365"}
assert not FORBIDDEN & set(FEATURES), "leakage-prone column in the feature list"

used_optional = [c for c in OPTIONAL_NUMERIC + OPTIONAL_CATEGORICAL if c in FEATURES]
print("Features:", FEATURES)
print("Optional (not yet in dictionary):", used_optional or "none")

In [ ]:
# Target distribution: prices are usually right-skewed with extreme outliers
df[TARGET].describe()

In [ ]:
# Decision: model log(price) and convert back for reporting.
# exp of a log-model prediction approximates the conditional median, which suits MAE.
# TODO: record the decision and reasoning in docs/
USE_LOG_TARGET = True

def to_model_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y

def from_model_target(y):
    return np.expm1(y) if USE_LOG_TARGET else y

## 4. Train / test split

A **grouped** split by listing id, so a listing that appears in several snapshots never lands in both train and test.

The split is saved to `evaluation/splits/` so `04_model_evaluation` uses exactly the same held-out cases.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(df, groups=df[ID_COL]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Sanity check: no listing identity on both sides
overlap = set(train_df[ID_COL]) & set(test_df[ID_COL])
assert not overlap, f"{len(overlap)} listing ids appear in both train and test"

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"train rows: {len(train_df):,} ({train_df[ID_COL].nunique():,} listings)  "
      f"test rows: {len(test_df):,} ({test_df[ID_COL].nunique():,} listings)")

In [ ]:
# Persist the split for reproducibility and for 04_model_evaluation
import hashlib

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "data_source": DATA_SOURCE,
    "synthetic": USE_SYNTHETIC,
    "seed": RANDOM_SEED,
    "train_ids": sorted(int(i) for i in train_df[ID_COL].unique()),
    "test_ids": sorted(int(i) for i in test_df[ID_COL].unique()),
}
manifest["sha256"] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
SPLIT_VERSION = manifest["sha256"][:12]

split_name = "split_synthetic.json" if USE_SYNTHETIC else "split_v1.json"
(SPLITS_DIR / split_name).write_text(json.dumps(manifest, indent=1))
print(f"saved {split_name} ({SPLIT_VERSION})")

## 5. Baseline: training-set median by neighbourhood and room type

The charter baseline: median nightly listing price per (`neighbourhood_cleansed`, `room_type`), computed on **training data only**, with a global-median fallback for unseen combinations.

The charter's primary target is that the learned model reduces held-out MAE by **at least 10%** versus this baseline.

In [ ]:
class MedianGroupBaseline:
    def __init__(self, group_cols):
        self.group_cols = group_cols

    def fit(self, X, y):
        data = X[self.group_cols].assign(_y=np.asarray(y))
        self.group_medians_ = data.groupby(self.group_cols)["_y"].median()
        self.global_median_ = float(np.median(y))
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_frame(X[self.group_cols])
        preds = self.group_medians_.reindex(keys).to_numpy()
        return np.where(np.isnan(preds), self.global_median_, preds)

    def is_fallback(self, X):
        keys = pd.MultiIndex.from_frame(X[self.group_cols])
        return np.isnan(self.group_medians_.reindex(keys).to_numpy())

# TODO: move to src/models/ once stable so other notebooks can import it
BASELINE_GROUPS = ["neighbourhood_cleansed", "room_type"]
baseline = MedianGroupBaseline(BASELINE_GROUPS).fit(X_train, y_train)

## 6. Learned models

Preprocessing lives inside the pipeline, so imputation, scaling and encoding are fitted on training data only.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

In [ ]:
from sklearn.base import clone

models = {
    "ridge": Pipeline([
        ("prep", clone(preprocessor)),
        ("model", Ridge(alpha=1.0)),
    ]),
    "hist_gbm": Pipeline([
        ("prep", clone(preprocessor)),
        ("model", HistGradientBoostingRegressor(random_state=RANDOM_SEED)),
    ]),
    # TODO: add candidates (e.g. LightGBM) if the team agrees on the dependency
}

for name, pipe in models.items():
    pipe.fit(X_train, to_model_target(y_train))
    print(f"fitted {name}")

### 6.1 Hyperparameter tuning (optional)

Tune with cross-validation **inside the training set only** (use `GroupKFold` on listing id). Never tune on the test set.

In [ ]:
# TODO: GridSearchCV / RandomizedSearchCV with GroupKFold(n_splits=5), groups=train_df[ID_COL]

## 7. Quick comparison on the held-out set

Headline metrics only, in the selected city's currency per night (not log space). Detailed error analysis, confidence intervals and the agent comparison belong in `04_model_evaluation`.

In [ ]:
def evaluate(name, predict_fn, X, y_true):
    start = time.perf_counter()
    y_pred = predict_fn(X)
    elapsed_ms = (time.perf_counter() - start) * 1000 / len(X)
    return {
        "approach": name,
        f"MAE ({CURRENCY}/night)": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "ms per prediction": elapsed_ms,
    }

results = [evaluate("baseline_median", baseline.predict, X_test, y_test)]
for name, pipe in models.items():
    results.append(evaluate(name, lambda X, p=pipe: from_model_target(p.predict(X)), X_test, y_test))

results_df = pd.DataFrame(results).round(2)
mae_col = f"MAE ({CURRENCY}/night)"
base_mae = results_df.loc[results_df["approach"] == "baseline_median", mae_col].iloc[0]
results_df["MAE reduction vs baseline (%)"] = (100 * (1 - results_df[mae_col] / base_mae)).round(1)
results_df["meets charter 10% target"] = results_df["MAE reduction vs baseline (%)"] >= 10
results_df

## 8. Initial error checks

Segment errors are required by the charter. Where does the model fail: particular neighbourhoods, room types or price ranges?

In [ ]:
best_name = "hist_gbm"  # TODO: pick based on results_df
best_model = models[best_name]

test_df["pred"] = from_model_target(best_model.predict(X_test))
test_df["residual"] = test_df[TARGET] - test_df["pred"]
test_df["abs_error"] = test_df["residual"].abs()

test_df.groupby("room_type")["abs_error"].agg(["mean", "median", "count"])

In [ ]:
test_df.groupby("neighbourhood_cleansed")["abs_error"].agg(["mean", "count"]) \
    .sort_values("mean", ascending=False).head(15)

In [ ]:
# TODO: residual vs actual plot, error by price decile, feature importance
# (save figures to reports/figures/)

## 9. Save model artifact, metadata and predictions

Artifacts go to `models/`, which is excluded from Git. The predictions follow the dictionary's **Predictions** table: listing identity, model version, predicted nightly price, residual and quality flags, in the same currency/night units as the target, with the split recorded.

In [ ]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)
version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
model_version = f"{best_name}_{version}"

artifact_path = MODELS_DIR / f"{model_version}.joblib"
joblib.dump(best_model, artifact_path)

metadata = {
    "model": best_name,
    "version": version,
    "synthetic_data": USE_SYNTHETIC,
    "city": CITY,
    "currency": CURRENCY,
    "snapshot_dates": sorted(df[SNAPSHOT_COL].dt.strftime("%Y-%m-%d").unique().tolist()),
    "data_source": DATA_SOURCE,
    "split_file": split_name,
    "split_version": SPLIT_VERSION,
    "target": TARGET,
    "target_unit": f"{CURRENCY}/night",
    "log_target": USE_LOG_TARGET,
    "features": FEATURES,
    "optional_features_not_in_dictionary": used_optional,
    "random_seed": RANDOM_SEED,
    "n_train": len(train_df),
    "n_test": len(test_df),
    "metrics": results_df.to_dict(orient="records"),
}
(MODELS_DIR / f"{model_version}.json").write_text(json.dumps(metadata, indent=2, default=str))

# Predictions table (dictionary schema)
predictions = pd.DataFrame({
    ID_COL: test_df[ID_COL].to_numpy(),
    SNAPSHOT_COL: test_df[SNAPSHOT_COL].dt.strftime("%Y-%m-%d").to_numpy(),
    "model_version": model_version,
    "split_version": SPLIT_VERSION,
    "predicted_price": test_df["pred"].round(2).to_numpy(),
    "observed_price": test_df[TARGET].to_numpy(),
    "residual": test_df["residual"].round(2).to_numpy(),
    "unit": f"{CURRENCY}/night",
    "flag_unseen_baseline_group": baseline.is_fallback(X_test),
    "flag_missing_core_feature": test_df[CORE_NUMERIC + CORE_CATEGORICAL].isna().any(axis=1).to_numpy(),
})
predictions.to_csv(MODELS_DIR / f"{model_version}_test_predictions.csv", index=False)
print(f"saved {artifact_path.name}, metadata and {len(predictions):,} test predictions")

## 10. Open decisions and TODOs

- [ ] When the first real snapshot is processed: set `USE_SYNTHETIC = False`, fill `CITY` and `CURRENCY` from `data/sources.md`
- [ ] With the data owner: add the optional features (beds, minimum_nights, reviews, property_type) to `data_dictionary.md`, or drop them
- [ ] Confirm that `02_preprocessing` outputs `snapshot_date` and parses `price` and `bathrooms_text` (this notebook handles both as a fallback)
- [ ] Decide on outlier handling for price (threshold from training data only) and document it
- [ ] Move `MedianGroupBaseline`, the parsers and pipeline builders into `src/models/`
- [ ] Choose and pin modelling libraries in `requirements.txt`
- [ ] Hyperparameter tuning with grouped cross-validation
- [ ] Prediction intervals for the agent and investor output
- [ ] Fill in the README evaluation table once real results exist